Install Libraries and Dependencies

In [ ]:
%pip install transformers datasets torch accelerate -q

In [ ]:
import pandas as pd
import re
import string
from datasets import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, f1_score, recall_score, confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
import torch
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
import torch.nn.functional as F
import os
import shutil
import time

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Configuration

In [ ]:
DEPRESSION_DATASET = 'dataset.csv'
EMOTION_DATASET = 'Twitter_Emotion_Dataset.csv'

In [ ]:
TEACHER_MODEL_PATH = 'saved_teacher_model'
FINAL_MODEL_PATH = 'final_depression_model'

In [ ]:
TEMP_TEACHER_PATH = 'local_teacher_model'
TEMP_FINAL_PATH = 'local_final_model'

In [ ]:
os.makedirs(os.path.dirname(TEACHER_MODEL_PATH), exist_ok=True)
os.makedirs(os.path.dirname(FINAL_MODEL_PATH), exist_ok=True)

In [ ]:
TRAIN_TEACHER = False # TRUE for initial training
TEACHER_MODEL_NAME = "indolem/indobertweet-base-uncased"

In [ ]:
def copy_from_drive_to_local(drive_path, local_path, description="model"):
    """Copy model from Drive to local storage with progress"""
    if os.path.exists(local_path):
        print(f"{description} already cached locally")
        return True

    if not os.path.exists(drive_path):
        print(f"{description} not found in Drive at {drive_path}")
        return False

    print(f"Copying {description} from Drive to local storage...")
    print(f"Source: {drive_path}")
    print(f"Target: {local_path}")

    total_size = sum(os.path.getsize(os.path.join(dirpath, filename))
                     for dirpath, _, filenames in os.walk(drive_path)
                     for filename in filenames) / (1024**2)
    print(f"Size: {total_size:.1f} MB")

    start_time = time.time()
    shutil.copytree(drive_path, local_path)
    elapsed = time.time() - start_time

    print(f"Copied in {elapsed:.1f} seconds")
    return True

def save_to_both_locations(model, tokenizer, drive_path, local_path, description="model"):
    """Save model to both Drive and local """
    print(f"\nSaving {description}...")

    print(f"Saving to local cache...")
    model.save_pretrained(local_path)
    tokenizer.save_pretrained(local_path)
    print(f"Saved to {local_path}")

    print(f"Saving to Google Drive...")
    model.save_pretrained(drive_path)
    tokenizer.save_pretrained(drive_path)
    print(f"Saved to {drive_path}")
    print(f"{description} saved successfully!")

def load_model_optimized(drive_path, local_path, model_name, description="model"):
    """Load model from local cache, or copy from Drive if needed"""
    if os.path.exists(local_path):
        print(f"Loading {description} from local cache (fast)")
        return AutoModelForSequenceClassification.from_pretrained(local_path)

    if copy_from_drive_to_local(drive_path, local_path, description):
        print(f"Loading {description} from local cache")
        return AutoModelForSequenceClassification.from_pretrained(local_path)

    print(f"{description} not found - will train from scratch")
    return None

Preprocessing

In [ ]:
# Cleaning Data

def clean_text(text):
  if pd.isna(text):
    return ''
  text = str(text)
  text = re.sub(r'http\S+', '', text)
  text = re.sub(r'\d+', '', text)
  text = text.translate(str.maketrans('', '', string.punctuation))
  text = re.sub(r'@\w+', '', text)
  text = re.sub(r'#(\w+)', r'\1', text)
  text = re.sub(r'\s+', ' ', text)
  text = re.sub(r'\n', ' ', text)
  text = re.sub(r'\[.*?\]', '', text)
  text = re.sub(r'[^\x00-\x7F]+', '', text)
  text = text.lower().strip()
  return text

In [ ]:
# Load Dataset

df = pd.read_csv(DEPRESSION_DATASET)
df = df[['full_text', 'label']].rename(columns={'full_text': 'text'})
df['text'] = df['text'].apply(clean_text)
df = df[df['text'].str.len() > 0]

df.head()

,text,label
0,diinorawrrrr capek hidup kek gini,1
1,punya orang tua satu nya tapi gak pernah bersy...,1
2,sandikara tanyakanrl h iyaa thn lalu aku udah ...,1
3,capek banget sama manusia yg merasa paking ben...,1
4,ucokniges pasti capek bangettt oi,1


In [ ]:
# Load Twitter Emotion Dataset

df_emotion = pd.read_csv(EMOTION_DATASET)
df_emotion['label'] = df_emotion['label'].apply(lambda x: 1 if x in ['anger', 'sadness', 'fear'] else 0)
df_emotion = df_emotion.rename(columns={'tweet': 'text'})
df_emotion['text'] = df_emotion['text'].apply(clean_text)
df_emotion = df_emotion[df_emotion['text'].str.len() > 0]

df_emotion.head()

,label,text
0,1,soal jln jatibarupolisi tdk bs gertak gubernur...
1,1,sesama cewe lho kayaknya harusnya bisa lebih r...
2,0,kepingin gudeg mbarek bu hj amad foto dari goo...
3,1,jln jatibarubagian dari wilayah tn abangpengat...
4,0,sharing pengalaman aja kemarin jam batalin tik...


In [ ]:
# Split Dataset

train_df, test_df = train_test_split(df_emotion, test_size=0.2, stratify=df_emotion['label'], random_state=42)

In [ ]:
!pip install transformers --upgrade

In [ ]:
# Load Tokenizer

if os.path.exists(TEMP_TEACHER_PATH):
    tokenizer = AutoTokenizer.from_pretrained(TEMP_TEACHER_PATH)
    print("Loaded tokenizer from local cache")
elif os.path.exists(TEACHER_MODEL_PATH):
    tokenizer = AutoTokenizer.from_pretrained(TEACHER_MODEL_PATH)
    print("Loaded tokenizer from Drive")
else:
    tokenizer = AutoTokenizer.from_pretrained(TEACHER_MODEL_NAME)
    print("Loaded tokenizer from HuggingFace")

Loaded tokenizer from Drive


In [ ]:
# Tokenizing

def tokenize(batch):
    return tokenizer(batch['text'], truncation=True, padding='max_length', max_length=128)

emo_train_ds = Dataset.from_pandas(train_df[['text', 'label']].reset_index(drop=True))
emo_test_ds = Dataset.from_pandas(test_df[['text', 'label']].reset_index(drop=True))
your_ds = Dataset.from_pandas(df[['text', 'label']].reset_index(drop=True))

train_tokenized = emo_train_ds.map(tokenize, batched=True).remove_columns(['text'])
test_tokenized = emo_test_ds.map(tokenize, batched=True).remove_columns(['text'])
your_tokenized = your_ds.map(tokenize, batched=True).remove_columns(['text'])

train_tokenized.set_format("torch")
test_tokenized.set_format("torch")
your_tokenized.set_format("torch")

Train Teacher Model

In [ ]:
teacher_model = load_model_optimized(TEACHER_MODEL_PATH, TEMP_TEACHER_PATH, TEACHER_MODEL_NAME, "teacher model")

In [ ]:
if TRAIN_TEACHER:
    if teacher_model is None:
        print("Teacher model not found, TRAIN_TEACHER is True. Training a new teacher model...")
        teacher_model = AutoModelForSequenceClassification.from_pretrained(TEACHER_MODEL_NAME, num_labels=2)
    else:
        print("Teacher model found, but TRAIN_TEACHER is True. Re-training the teacher model...")

    training_args = TrainingArguments(
        output_dir='/content/teacher_results',
        num_train_epochs=3,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        eval_strategy="epoch",
        logging_strategy="steps",
        logging_steps=50,
        save_strategy="epoch",
        learning_rate=2e-5,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        report_to="none",
        fp16=True
    )

    trainer = Trainer(
        model=teacher_model,
        args=training_args,
        train_dataset=train_tokenized,
        eval_dataset=test_tokenized
    )

    trainer.train()

    save_to_both_locations(teacher_model, tokenizer, TEACHER_MODEL_PATH, TEMP_TEACHER_PATH, "teacher model")
    print("Next time, set TRAIN_TEACHER = False to skip this step!")

else:
    if teacher_model is None:
        raise ValueError(
            "Teacher model not found and TRAIN_TEACHER is False. "
            "Set TRAIN_TEACHER = True to train the model, or ensure it exists in Drive."
        )
    else:
        print("Teacher model loaded successfully (TRAIN_TEACHER is False, so skipping training).")

Teacher model loaded successfully (TRAIN_TEACHER is False, so skipping training).


In [ ]:
# Extract Teacher Logits

device = "cuda" if torch.cuda.is_available() else "cpu"
teacher_model.eval()
teacher_model.to(device)

def get_logits(dataset, desc="Extracting logits"):
    """Extract logits from teacher model with progress bar"""
    dataloader = DataLoader(dataset, batch_size=32)
    outputs_list = []

    for batch in tqdm(dataloader, desc=desc):
        batch = {k: v.to(device) for k, v in batch.items() if k in ['input_ids', 'attention_mask']}

        with torch.no_grad():
            logits = teacher_model(**batch).logits

        outputs_list.append(logits.cpu())

    return torch.cat(outputs_list)

print("Extracting teacher logits for depression dataset...")
teacher_logits = get_logits(your_tokenized, "Teacher predictions (train)")
print(f" Teacher logits shape: {teacher_logits.shape}")

if 'teacher_logits' not in your_tokenized.column_names:
    your_tokenized = your_tokenized.add_column("teacher_logits", teacher_logits.tolist())
    print(" Teacher logits added to training dataset")
else:
    print(" Teacher logits already exist in training dataset, skipping addition.")

print("\nExtracting teacher logits for test dataset...")
test_teacher_logits = get_logits(test_tokenized, "Teacher predictions (test)")
print(f" Test teacher logits shape: {test_teacher_logits.shape}")

if 'teacher_logits' not in test_tokenized.column_names:
    test_tokenized = test_tokenized.add_column("teacher_logits", test_teacher_logits.tolist())
    print(" Teacher logits added to test dataset")
else:
    print(" Teacher logits already exist in test dataset, skipping addition.")

del teacher_model
torch.cuda.empty_cache()

Train Student Model (Knowledge Distillation)

In [ ]:
student_model = AutoModelForSequenceClassification.from_pretrained(TEACHER_MODEL_NAME, num_labels=2)

def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)

    return{
        'accuracy': accuracy_score(labels, preds),
        'recall': recall_score(labels, preds, average='weighted', zero_division=0),
        'f1': f1_score(labels, preds, average='weighted', zero_division=0),
        'precision': precision_score(labels, preds, average='weighted', zero_division=0)
    }

class DistillationTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop("labels")
        teacher_logits = torch.tensor(inputs.pop("teacher_logits")).to(self.args.device)

        outputs = model(**inputs)
        student_logits = outputs.logits

        ce_loss = F.cross_entropy(student_logits, labels)

        kl_loss = F.kl_div(
            F.log_softmax(student_logits/2, dim=-1),
            F.softmax(teacher_logits/2, dim=-1),
            reduction="batchmean"
        ) * (2 * 2)

        loss = 0.5 * ce_loss + 0.5 * kl_loss
        return (loss, outputs) if return_outputs else loss

student_args = TrainingArguments(
    output_dir="/content/student_results",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=50,
    learning_rate=2e-5,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    report_to="none",
    fp16=True,
    remove_unused_columns=False
)

student_trainer = DistillationTrainer(
    model=student_model,
    args=student_args,
    train_dataset=your_tokenized,
    eval_dataset=test_tokenized,
    compute_metrics = compute_metrics
)

student_trainer.train()

Evaluate Model

In [ ]:
predictions = student_trainer.predict(test_tokenized)
pred_labels = predictions.predictions.argmax(-1)
true_labels = predictions.label_ids

eval_results = student_trainer.evaluate()
print(f"Accuracy: {eval_results['eval_accuracy']}")
print(f"F1 Score: {eval_results['eval_f1']}.")
print(f"Precision: {eval_results['eval_precision']}")
print(f"Recall: {eval_results['eval_recall']}")
print("\nClassification Report")
print(classification_report(true_labels, pred_labels,
                            target_names=['No Depression', 'Depression']))

cm = confusion_matrix(true_labels, pred_labels)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['No Depression', 'Depression'], yticklabels=['No Depression', 'Depression'])
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix')
plt.show()

In [ ]:
# Save Final Model

save_to_both_locations(student_model, tokenizer, FINAL_MODEL_PATH, TEMP_FINAL_PATH, "final model")

Test Model

In [ ]:
def predict_depression(text):
  model_path = TEMP_FINAL_PATH if os.path.exists(TEMP_FINAL_PATH) else FINAL_MODEL_PATH
  model = AutoModelForSequenceClassification.from_pretrained(FINAL_MODEL_PATH)
  tokenizer = AutoTokenizer.from_pretrained(model_path)
  model.eval()

  if torch.cuda.is_available():
    model = model.cuda()

  cleaned_text = clean_text(text)
  inputs = tokenizer(cleaned_text, return_tensors="pt", truncation=True, max_length=128)

  if torch.cuda.is_available():
    inputs = {k: v.cuda() for k, v in inputs.items()}

  with torch.no_grad():
    outputs = model(**inputs)
    probabilities = torch.softmax(outputs.logits, dim=-1)[0]
    prediction = torch.argmax(outputs.logits, dim=-1).item()
    confidence = probabilities[prediction].item()

    return {
        "depression_detected": bool(prediction),
        "confidence": f"{confidence:.2%}",
        "label": "Depression" if prediction == 1 else "No Depression",
        "probabilities": {
            "no_depression": f"{probabilities[0].item():.2%}",
            "depression": f"{probabilities[1].item():.2%}"
        }
    }

In [ ]:
test_tweets = [
    "Hari ini gue sedih banget",
    "Seru bgt bisa hangout sm temen-temen",
    "rasanya hampa"
]

In [ ]:
for i, tweet in enumerate(test_tweets):
  result = predict_depression(tweet)
  print(f"{i+1}. Tweet: '{tweet}'")
  print(f"Result: {result['label']} (Confidence: {result['confidence']})")
  print(f"Probabilities: {result['probabilities']}")

Deploy Model

In [ ]:
import gradio as gr
import os

gradio_model_path = TEMP_FINAL_PATH if os.path.exists(TEMP_FINAL_PATH) else FINAL_MODEL_PATH

model = AutoModelForSequenceClassification.from_pretrained(gradio_model_path)
tokenizer = AutoTokenizer.from_pretrained(gradio_model_path)

def predict(tweet):
    model.eval()
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model.to(device)

    cleaned_text = clean_text(tweet)
    inputs = tokenizer(cleaned_text, return_tensors="pt", truncation=True, max_length=128)

    if torch.cuda.is_available():
        inputs = {k: v.cuda() for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)
        probabilities = torch.softmax(outputs.logits, dim=-1)[0]
    return {
        "Depression": float(probabilities[1]),
        "No Depression": float(probabilities[0])
    }

demo = gr.Interface(
    fn=predict,
    inputs=gr.Textbox(label="Enter tweet (in Indonesian)"),
    outputs=gr.Label(label="Prediction"),
    title="Indonesian Tweet Depression Detection",
    article="""
    <div style='margin-top=20px;'>
    <b>Disclaimer:</b> The predicted results CANNOT be used to diagnose depression or any mental illness.
    This application is for research purposes only.
    If you or someone you know is experiencing mental health difficulties, please seek professional help.
    </div>
    </div>
    """
)

demo.launch(share=True)